Note: this nootbook if for preforming BERTTOpic clustering when the data is too much big to fit in memory leading to Oout of memory (OOM). The nootbook dideivde the data to chunks and  fit a sepreate berttopic model on eeach of the cunnkks. 

# Imports

In [ ]:
# ML
import pandas as pd
import numpy as np
import pyarrow  # ensures pandas can read/write parquet
import math

# Models
from bertopic import BERTopic
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import torch

# System
import gc
from pathlib import Path
from datetime import datetime

# import subprocess
import sys
import fcntl
import tempfile
import os


# Pyton
from functools import wraps
from typing import Iterator, List, Tuple

# Visualization
import matplotlib.pyplot as plt
from IPython.display import display
from pprint import pprint
import matplotlib.ticker as ticker

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Parameters

In [ ]:
is_test = False

In [ ]:
country="UAE"
input_data_path = f"../results/Labeled_Datasets/{country}/{country}_part_all/step_1_translate_topics_entities/{country}_part_all_step_1.gzip.parquet"
min_topic_size = 2**8
chunk_size = 2**17

if not is_test:
    print(f"Input data path: {input_data_path}")
    print(f"Minimum topic size: {min_topic_size}")
    print(f"Chunk size: {chunk_size:,}")

In [ ]:
if is_test:
    print("Running in test mode. Only a small subset of the data will be processed.")
    min_topic_size = 10
    chunk_size = 2**7
    print(f"min_topic_size = {min_topic_size}, chunk_size = {chunk_size}")

# Loading and Exporting


In [ ]:
from pathlib import Path


def get_dataset_part_path(path: str | Path) -> Path:
    """Return the dataset-part directory containing a pipeline step.

    Args:
        path: Input file or directory inside a ``step_*`` directory.

    Returns:
        Parent directory of the nearest ``step_*`` ancestor.

    Raises:
        ValueError: If the path is not inside a pipeline step directory.
    """
    input_path = Path(path).expanduser()
    for candidate in (input_path, *input_path.parents):
        if candidate.name.startswith("step_"):
            return candidate.parent
    raise ValueError("input_data_path must be inside a step_* directory")

In [ ]:
input_data_path = Path(input_data_path).expanduser()
topic_col = f"BERTopic_topic_{min_topic_size}"
topic_top_words_col = f"{topic_col}_top_words"
dataset_part_path = get_dataset_part_path(input_data_path)
dataset_part = dataset_part_path.name
output_path = dataset_part_path / "step_2_BERTopic_clustering"
if is_test:
    output_path = output_path / "test"
output_path = output_path / f"topic_col_{topic_col}"
output_path.mkdir(parents=True, exist_ok=True)
output_data_path = output_path.parent / f"{dataset_part}_step_2.gzip.parquet"

print(f"Input data path: {input_data_path}")
print(f"Output path: {output_path}")
print(f"Output data path: {output_data_path}")

In [ ]:
import pyarrow.parquet as pq

schema = pq.read_schema(input_data_path)

print(f"\nSchema of input data: {input_data_path}")
print("-" * 80)
print(f"{'Column':<40} {'Type'}")
print("-" * 80)

for field in schema:
    print(f"{field.name:<40} {field.type}")

print("-" * 80)
print(f"Total columns: {len(schema)}\n")

In [ ]:
required_columns = ["translated_post", "accountid", "post_time", "is_control"]
print(f"Required columns: {required_columns}")

if is_test:
    parquet_file = pq.ParquetFile(input_data_path)
    batch = next(parquet_file.iter_batches(batch_size=1_000, columns=required_columns))
    df = batch.to_pandas()
else:
    df = pd.read_parquet(input_data_path, columns=required_columns, engine="pyarrow")

In [ ]:
print(f"df shape: {df.shape}")
print("df columns:")
pprint(list(df.columns))

In [ ]:
display(df.head())

# Utils

## Logs

In [ ]:
REPORT_PATH = output_path / "progress_report.txt"

def write_report(msg: str):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"{timestamp} | {msg}"

    print(line)
    with open(REPORT_PATH, "a") as f:
        f.write(line + "\n")


write_report(f"Input data path: {input_data_path}")
write_report(f"Output path: {output_path}")

def report_done(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        try:
            result = func(*args, **kwargs)
            write_report(f"Completed: {func.__name__}")
            return result
        except Exception as e:
            write_report(f"FAILED: {func.__name__} | {e}")
            raise
    return wrapper

In [ ]:
import os
import platform
import subprocess
from pathlib import Path

import psutil
import torch

allocated_cpu_ids = sorted(os.sched_getaffinity(0))

cpu_info = subprocess.run(
    ["lscpu"],
    capture_output=True,
    text=True,
    check=True,
).stdout

cpu_model_name = "Unknown"

for line in cpu_info.splitlines():
    if line.startswith("Model name:"):
        cpu_model_name = line.split(":", 1)[1].strip()
        break

hardware_lines = [
    "Hardware and Slurm allocation",
    "-" * 75,
    f"Hostname:               {platform.node()}",
    f"CPU model:              {cpu_model_name}",
    f"Node physical cores:    {psutil.cpu_count(logical=False)}",
    f"Node logical CPUs:      {psutil.cpu_count(logical=True)}",
    f"Allocated CPUs:         {len(allocated_cpu_ids)}",
    f"Allocated CPU IDs:      {allocated_cpu_ids}",
    f"Slurm job ID:           {os.getenv('SLURM_JOB_ID', 'Not running under Slurm')}",
]

if torch.cuda.is_available():
    hardware_lines.extend([
        f"Allocated GPUs:         {torch.cuda.device_count()}",
        f"CUDA version:           {torch.version.cuda}",
    ])

    for gpu_index in range(torch.cuda.device_count()):
        properties = torch.cuda.get_device_properties(gpu_index)

        hardware_lines.extend([
            f"GPU {gpu_index} name:          {properties.name}",
            f"GPU {gpu_index} memory:        {properties.total_memory / 1024**3:.2f} GB",
            f"GPU {gpu_index} capability:    {properties.major}.{properties.minor}",
        ])
else:
    hardware_lines.append("Allocated GPUs:         0")

hardware_lines.append("-" * 75)

hardware_report = "\n".join(hardware_lines)

print(hardware_report)

hardware_report_path = Path(output_path) / "hardware_info.txt"
_ = hardware_report_path.write_text(hardware_report, encoding="utf-8")

write_report(f"Wrote hardware report to: {hardware_report_path}")

Hardware and Slurm allocation
----------------------------------------------------------------------
Hostname:               ise-cpu256-31.auth.ad.bgu.ac.il
Node physical cores:    128
Node logical CPUs:      256
Allocated CPUs:         24
Allocated CPU IDs:      [61, 91, 92, 99, 100, 117, 118, 119, 120, 121, 122, 123, 189, 219, 220, 227, 228, 245, 246, 247, 248, 249, 250, 251]
SLURM_CPUS_PER_TASK:    Not set
SLURM_CPUS_ON_NODE:     Not set
SLURM_JOB_CPUS_PER_NODE:Not set
Slurm job ID:           19483174
Slurm node list:        Not set
GPU available:          False
----------------------------------------------------------------------
2026-07-21 20:33:00 | Wrote hardware report to: ../results/Labeled_Datasets/UAE/UAE_part_all/step_2_BERTopic_clustering/topic_col_BERTopic_topic_256/hardware_info.txt (642 characters)


In [ ]:
write_report(" ############# running: big_data_topic_clustering.ipynb ############# ")
write_report(f"Today date: {datetime.now().strftime('%Y-%m-%d')}")
write_report(f"Input data path: {input_data_path}")
write_report(f"Output path: {output_path}")
write_report(f"Loaded df: {input_data_path.name}.")
write_report(f"df shape: {df.shape}")
write_report(f"min_topic_size: {min_topic_size}")
print("df head:")
display(df.head(3))

## Save/Update df

In [ ]:
# @report_done
# def save_df(
#         df: pd.DataFrame,
#         output_data_path: Path = output_data_path
#     )-> None :
#     df.to_parquet(output_data_path, index=False, compression="gzip")
#     write_report(f"df is saved to: {output_data_path}")

In [ ]:
from __future__ import annotations

@report_done
def update_df_collumns_atomically(
    df: pd.DataFrame,
    collums_to_update: str | list[str],
    output_data_path: Path = output_data_path,
    compression: str = "gzip",
) -> pd.DataFrame:
    """Safely update or add columns in a shared Parquet file.
    This assumes the same row order/length in df and df_from_file. 

    Performs a transaction-style update:
    exclusive lock -> read latest -> update columns -> write temp -> os.replace.

    Args:
        df: DataFrame containing the new column values.
        collums_to_update: Column name or list of column names to update/add.
        output_data_path: Directory containing the Parquet file.
        compression: Parquet compression codec. Default is "gzip".

    Returns:
        The updated DataFrame that was written to disk.

    Raises:
        RuntimeError: If reading, updating, or writing fails. Original exception
            is attached as context.
    """

    write_report(f"[update_df_collumns_atomically]\n Updating columns: {collums_to_update} \nfrom df to {output_data_path}")
    lock_path = output_data_path.with_suffix(output_data_path.suffix + ".lock")

    if isinstance(collums_to_update, str):
        collums_to_update = [collums_to_update]

    tmp_path: Path | None = None

    with open(lock_path, "w") as lockf:
        fcntl.flock(lockf, fcntl.LOCK_EX)
        try:
            if not output_data_path.exists():
                df_from_file = df.copy()
            else:
                df_from_file = pd.read_parquet(output_data_path)

            for col in collums_to_update:
                df_from_file[col] = df[col]

            fd, tmp = tempfile.mkstemp(
                dir=str(output_data_path.parent),
                prefix=output_data_path.name + ".",
                suffix=".tmp",
            )
            os.close(fd)
            tmp_path = Path(tmp)

            df_from_file.to_parquet(tmp_path, index=False, compression=compression)
            os.replace(tmp_path, output_data_path)

            write_report("[update_df_collumns_atomically]\n Updated columns: " + ", ".join(collums_to_update) + f" \nsave to {output_data_path}")

            return df_from_file

        except Exception as e:
            raise RuntimeError(f"Failed to update parquet file: {output_data_path}") from e

        finally:
            if tmp_path is not None and tmp_path.exists():
                tmp_path.unlink(missing_ok=True)
            fcntl.flock(lockf, fcntl.LOCK_UN)

# Posting Same Topic

In [ ]:
time_windows = ["12H", "3D"]

In [ ]:
from collections import defaultdict, deque

@report_done
def add_posting_same_topic(
    df: pd.DataFrame,
    topic_column: str,
    time_window: str = "3D",
    account_id_column: str = "accountid",
    timestamp_column: str = "post_time",
) -> str:
    """Add prior authors posting the same topic within a time window.

    The DataFrame is modified in place. Author IDs are unique and ordered by
    the timestamps of their matching posts, from oldest to newest.

    Returns:
        Name of the added account-ID column.
    """
    required_columns = {topic_column, account_id_column, timestamp_column}
    missing_columns = sorted(required_columns - set(df.columns))
    if missing_columns:
        raise KeyError(f"Missing required columns: {missing_columns}")

    output_column = f"posting_same_topic_{topic_column}_{time_window}_accountids"
    timestamps = pd.to_datetime(df[timestamp_column], errors="coerce").to_numpy(
        dtype="datetime64[ns]"
    )
    topics = df[topic_column].to_numpy()
    account_ids = df[account_id_column].to_numpy()
    window_delta = np.timedelta64(pd.Timedelta(time_window).value, "ns")

    output_account_ids = [[] for _ in range(len(df))]
    topic_history: dict[object, deque] = defaultdict(deque)

    for current_index in np.argsort(timestamps, kind="stable"):
        current_timestamp = timestamps[current_index]
        current_topic = topics[current_index]
        if np.isnat(current_timestamp) or pd.isna(current_topic):
            continue

        topic_posts = topic_history[current_topic]
        while topic_posts and current_timestamp - topic_posts[0][0] > window_delta:
            topic_posts.popleft()

        output_account_ids[current_index] = list(
            dict.fromkeys(account_ids[post_index] for _, post_index in topic_posts)
        )
        topic_posts.append((current_timestamp, current_index))

    df[output_column] = output_account_ids
    write_report(
        f"[add_posting_same_topic] Added {output_column} for topic column "
        f"{topic_column} and time window {time_window}."
    )
    return output_column

In [ ]:
# Called after BERTopic creates the topic column.

# BERT English Topic Clustering

Note: the topic clustering doent influenced by the entities since it is preformed on the text without the entitites.

## Utils

### Posts Topic Stats

In [ ]:
@report_done
def make_topic_stats(df, topic_col_name):
    """
    Calculate per-topic post counts.

    Args:
        df (pd.DataFrame): Data with topics and 'is_control'.
        topic_col_name (str): Column name of topic labels.

    Returns:
        topic_stats (pd.DataFrame): Colums: [topic, total_count, io_Count, legitimate_posts, io_percent].
    """
    topic_stats = (
        df.groupby(topic_col_name)
          .agg(
              total_count=(topic_col_name, "count"),
              io_count=("is_control", lambda x: (~x).sum())
          )
          .reset_index()
    )
    topic_stats["legitimate_posts"] = topic_stats["total_count"] - topic_stats["io_count"]
    topic_stats["io_percent"] = topic_stats["io_count"] / topic_stats["total_count"] * 100
    return topic_stats

In [ ]:
from matplotlib.lines import Line2D

def plot_topic_stats(topic_stats, topic_col_name, output_path: str | Path = ""):
    """
    Plot stacked IO vs legitimate post counts per topic.
    Saves the fig to output_path if provided.

    Args:
        topic_stats (pd.DataFrame): Output of make_topic_stats().
        topic_col_name (str): Topic column to plot.
        output_path (str): Path to *folder* save the plot.

    Returns:
        None
    """
    topics = topic_stats[topic_col_name]
    io_count = topic_stats["io_count"]
    legitimate_count = topic_stats["legitimate_posts"]
    io_percent = topic_stats["io_percent"]

    plt.figure(figsize=(12, 6))

    # 🔴 Bottom: IO posts
    bar_io = plt.bar(topics, io_count, label="IO posts", color="red")

    # 🔵 Top: Non-IO posts stacked above IO
    bar_legit = plt.bar(topics, legitimate_count, bottom=io_count, label="Non-IO posts")

    # Add IO percentage text
    ax = plt.gca()
    for x, io, legit, pct in zip(topics, io_count, legitimate_count, io_percent):
        total_height = io + legit
        ax.text(
            x,
            total_height + 0.5,              # slightly above the bar
            f"{pct:.1f}%",                 # format like 23.5%
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="regular"
        )

    plt.title(f"Post Count per Topic ({topic_col_name})")
    plt.xlabel("Topic ID")
    plt.ylabel("Number of Posts")
    plt.xticks(topics)
    plt.grid(axis="y")
    plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))

    percent_legend = Line2D([], [], color="none")
    total_posts_legend = Line2D([], [], color="none")
    
    plt.legend(
        handles=[bar_io, bar_legit, percent_legend, total_posts_legend],
        labels=[
            "IO posts",
            "Non-IO posts",
            "% IO Posts Ratio",
            f"Total posts: {topic_stats['total_count'].sum():,}",
        ],
        loc="best",
    )


    plt.tight_layout()


    if output_path:
        output_path = Path(output_path)
        plot_file_path = output_path / (f"{topic_col_name}_topic_clustering.png")
        plt.savefig(plot_file_path, dpi=300)
        print(f"plot is saved to {plot_file_path}")

    plt.show()

    plt.close()

### Accoutns Topic Stats

In [ ]:
@report_done
def make_topic_account_stats(
    df: pd.DataFrame,
    topic_col: str,
    author_col: str = "accountid",
    is_control_col: str = "is_control",
    threshold: float = 0.5,
) -> pd.DataFrame:
    """Compute per-topic unique account counts (IO vs control) using an author-level threshold.

    Args:
        df: Post-level DataFrame.
        topic_col: Topic column name.
        author_col: Author identifier column.
        is_control_col: Column where 1 = control, 0 = IO.
        threshold: IO-rate threshold to label an author as IO (1 = IO).

    Returns:
        DataFrame with per-topic counts of unique IO/control accounts and IO percentage.
    """
    # Author-level IO rate: 1 - mean(is_control)
    author_io_rate = 1 - df.groupby(author_col)[is_control_col].mean()
    author_io_tag = (author_io_rate >= threshold).astype(int)

    topic_author_df = df[[topic_col, author_col]].drop_duplicates().copy()
    topic_author_df["author_io_tag"] = topic_author_df[author_col].map(author_io_tag)

    stats = (
        topic_author_df
        .groupby(topic_col)["author_io_tag"]
        .value_counts()
        .unstack(fill_value=0)
    )

    # Ensure both columns exist even if absent in data
    stats = stats.reindex(columns=[0, 1], fill_value=0).rename(
        columns={0: "control_accounts", 1: "io_accounts"}
    ).reset_index()

    stats["total_accounts"] = stats["control_accounts"] + stats["io_accounts"]
    stats["io_percent"] = (stats["io_accounts"] / stats["total_accounts"]).mul(100)

    return stats

In [ ]:
def plot_topic_account_stats(
    topic_account_stats: pd.DataFrame,
    topic_col_name: str,
    output_path: Path | str | None = None,
):
    """Plot IO vs control account counts per topic.

    Args:
        topic_account_stats: Output of make_topic_account_stats().
        topic_col_name: Topic column name.
        output_path: Optional folder to save the plot.

    Returns:
        None
    """
    topics = topic_account_stats[topic_col_name]
    io_count = topic_account_stats["io_accounts"]
    control_count = topic_account_stats["control_accounts"]
    io_percent = topic_account_stats["io_percent"]

    plt.figure(figsize=(12, 6))

    # 🔴 Bottom: IO accounts
    bar_io = plt.bar(topics, io_count, label="IO accounts", color="red")

    # 🔵 Top: Control accounts
    bar_control = plt.bar(
        topics,
        control_count,
        bottom=io_count,
        label="Control accounts",
    )

    # Add IO percentage text
    ax = plt.gca()
    for x, io, ctrl, pct in zip(topics, io_count, control_count, io_percent):
        total = io + ctrl
        ax.text(
            x,
            total + 0.3,
            f"{pct:.1f}%",
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="regular",
        )

    plt.title(f"Unique Accounts per Topic ({topic_col_name})")
    plt.xlabel("Topic ID")
    plt.ylabel("Number of Unique Accounts")
    plt.xticks(topics)
    plt.grid(axis="y")
    plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
    
    # Legend with extra info lines
    percent_legend = Line2D([], [], color="none")
    total_accounts_legend = Line2D([], [], color="none")

    plt.legend(
        handles=[bar_io, bar_control, percent_legend, total_accounts_legend],
        labels=[
            "IO accounts",
            "Control accounts",
            "% IO Accounts Ratio",
            f"Total accounts: {topic_account_stats['total_accounts'].sum():,}",
        ],
        loc="best",
    )

    if output_path:
        output_path = Path(output_path)
        plot_file_path = output_path / f"{topic_col_name}_topic_accounts.png"
        plt.savefig(plot_file_path, dpi=300)
        print(f"Plot saved to {plot_file_path}")

    plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
    plt.show()
    plt.close()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

post_text_col = "translated_post"

posts_texts = df[post_text_col].astype(str).tolist()

model = SentenceTransformer("digio/Twitter4SSE", device=device)
write_report(f"Embeddings Model loaded: {model.__class__.__name__} on device {device}")

embeddings = model.encode(
    posts_texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True, 
)

embeddings = np.ascontiguousarray(embeddings, dtype=np.float32)
write_report(f"Generated embeddings shape: {embeddings.shape}")

## BERTopic

[BERTopic Parametes](https://maartengr.github.io/BERTopic/getting_started/parameter%20tuning/parametertuning.html#calculate_probabilities)

In [ ]:
# topic_col is defined with the output path so both use the same value.
BERTopic_probs_col_name = f"BERTopic_prob_{min_topic_size}"

### Build Chunks Models

https://maartengr.github.io/BERTopic/getting_started/merge/merge.html

In [ ]:
# Make output folder for chunk models
import shutil

chunk_models_output_path = Path(output_path) / f"BERTopic_models_min_topic_size_{min_topic_size}"
# clean up any existing folder to avoid conflicts with previous runs
if chunk_models_output_path.exists():
    shutil.rmtree(chunk_models_output_path)
chunk_models_output_path.mkdir(exist_ok=True)

print(f"Chunk models will be saved to: {chunk_models_output_path}")


In [ ]:
def iter_text_chunks(
    texts: List[str],
    chunk_size: int,
) -> Iterator[Tuple[int, List[str]]]:
    """Yield (start_index, text_chunk) pairs of fixed size.

    Args:
        texts: List of documents.
        chunk_size: Max number of documents per chunk.

    Yields:
        Tuple[int, List[str]]: (start_index, chunk_texts).
    """
    for start in range(0, len(texts), chunk_size):
        yield start, texts[start : start + chunk_size]


@report_done
def build_and_save_chunk_models(
    texts: List[str],
    embeddings: np.ndarray,
    chunk_size: int,
    min_topic_size: int,
    output_dir: str | Path,
) -> None:
    """Train BERTopic models on chunks and save each to disk.

    Args:
        texts: Documents.
        embeddings: Precomputed embeddings (n_docs, dim).
        chunk_size: Max docs per chunk.
        min_topic_size: BERTopic min_topic_size.
        output_dir: Directory to save chunk models.
    """

    device = "cuda" if torch.cuda.is_available() else "cpu"
    write_report(f"Building and saving chunk models to {output_dir} with chunk size {chunk_size} and min_topic_size {min_topic_size} on device {device}")

    for idx, (start, text_chunk) in enumerate(iter_text_chunks(texts, chunk_size)):
        end = start + len(text_chunk)
        emb_chunk = embeddings[start:end]

        
        if device == "cuda": # gpu

            from cuml.cluster import HDBSCAN
            from cuml.manifold import UMAP
            torch.cuda.empty_cache()  # free GPU memory before training
            umap_model = UMAP(random_state=42)
            hdbscan_model = HDBSCAN(min_cluster_size=min_topic_size)
            model = BERTopic(
                language="english",
                umap_model=umap_model,
                hdbscan_model=hdbscan_model,
                low_memory=True,
                calculate_probabilities=False,
                min_topic_size=min_topic_size
            )

        else: # CPU 
            model = BERTopic(
                language="english",
                min_topic_size=min_topic_size,
                low_memory=True,
                calculate_probabilities=False,
            )

        model.fit(text_chunk, emb_chunk)

        model_path = output_dir / f"chunk_model_{idx}"
        model.save(model_path)
        write_report(f"Saved chunk model {idx} for texts {start} to {end} at {model_path}")

        # free RAM
        del model, emb_chunk, text_chunk # delete references
        gc.collect() # Garbage Collector

### Merge Chunks Models

In [ ]:
@report_done
def load_and_merge_BERTopic_models(
    chunk_models_dir: str | Path,
    min_similarity: float = 0.7, # default value is 0.7 in BERTopic docs
) -> BERTopic:
  
    chunk_models_dir = Path(chunk_models_dir)
    paths = sorted(chunk_models_dir.iterdir())
    
    if not paths:
        raise ValueError(f"No chunk models found in {chunk_models_dir}")
    
    merged_model  = None

    for p in paths:
        chunk_model = BERTopic.load(p)

        if merged_model is None: # first model
            merged_model = chunk_model
        else:
            merged_model = BERTopic.merge_models([merged_model, chunk_model], min_similarity=min_similarity)
            del chunk_model
            gc.collect()


    if merged_model is None:
        raise RuntimeError("merged_model is None after loading chunk models.")
    

    # save merged model
    merge_model_file_name = f"merged_BERTopic_model_min_similarity_{min_similarity}"
    merged_model.save(chunk_models_dir / merge_model_file_name)
    write_report(f"Merged model saved to {chunk_models_dir / merge_model_file_name}")

    return merged_model

### Run BERTopic Modeling

In [ ]:
build_and_save_chunk_models(
    texts=posts_texts,
    embeddings=embeddings,
    chunk_size=chunk_size,
    min_topic_size=min_topic_size,
    output_dir=chunk_models_output_path,
)

merge_BERTopic = load_and_merge_BERTopic_models(chunk_models_dir = chunk_models_output_path)

BERTopic_topic, BERT_probs = merge_BERTopic.transform(posts_texts, embeddings)

In [ ]:
# update df
df[topic_col] = BERTopic_topic
df[BERTopic_probs_col_name] = BERT_probs

# top words tuple
df[topic_top_words_col] = (
    df[topic_col]
    .map(merge_BERTopic.get_topic)
    .map(
        lambda topic_words: (
            tuple(word for word, _ in topic_words)
            if topic_words
            else ()
        )
    )
)

In [ ]:
# save as category to save memory
# df[topic_col] = df[topic_col].astype('category') # problem with saving parquet with category type
# df[topic_top_words_col] = df[topic_top_words_col].astype('category')  # problem with saving parquet with category type

### Update and Save df to Disk

Using file for lock to enable parallel instances.

In [ ]:
collum_to_update = [topic_col, BERTopic_probs_col_name, topic_top_words_col]

df = update_df_collumns_atomically(df, collum_to_update)

write_report("Saved df with BERTopic topics.")

## Posting Same Topic

In [ ]:
posting_same_topic_columns = [
    add_posting_same_topic(
        df=df,
        topic_column=topic_col,
        time_window=time_window,
    )
    for time_window in time_windows
]

df = update_df_collumns_atomically(df, posting_same_topic_columns)
write_report(
    "Saved posting-same-topic columns: " + ", ".join(posting_same_topic_columns)
)

## Validation

In [ ]:
for column in posting_same_topic_columns:
    assert len(df[column]) == len(df)
    assert df[column].map(lambda value: isinstance(value, list)).all()
    non_empty_count = df[column].map(bool).sum()
    print(f"{column}: {non_empty_count:,} posts with prior same-topic authors")

validation_columns = [
    "post_time",
    "accountid",
    topic_col,
    *posting_same_topic_columns,
]
matching_rows = df[df[posting_same_topic_columns].map(bool).any(axis=1)]
display(matching_rows[validation_columns].head())

### Graph and Statstistics

In [ ]:
try:
    fig = merge_BERTopic.visualize_topics(custom_labels=True)
    display(fig)   # show it
except Exception as e:
    print("Not enough topics for visualization")
    print("Error:", e)

In [ ]:
merge_BERTopic.get_topic_info()

Topic Post Statstistics

In [ ]:
BERT_topic_stats = make_topic_stats(df, topic_col)
plot_topic_stats(BERT_topic_stats, topic_col, output_path)

Topic Account Statstistics

In [ ]:
topic_account_stats = make_topic_account_stats(df, topic_col)
display(topic_account_stats)
plot_topic_account_stats(topic_account_stats, topic_col, output_path)